# 04 · Mechanism probes (bridge to *our* method)
The comparative paper should not stop at "who wins". These three probes test the FYP hypothesis from the literature review: **tokenisation granularity
and frozen-LLM bottlenecks hurt around sharp local dynamics**.  Needs `--save-predictions` runs from notebook 02.

In [ ]:
import os, sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, yaml
pd.set_option("display.width", 160, "display.max_columns", 30)

In [ ]:
from tscompare import aggregate as ag
from tscompare.analysis.mechanism import error_by_regime
from tscompare.analysis import plots
import matplotlib.pyplot as plt
RES = "results" if Path("results").exists() and any(Path("results").rglob("*.npz")) else "/tmp/probe_demo"
print(RES)

In [ ]:
# demo data so the notebook runs without GPU results
if RES == "/tmp/probe_demo":
    from tscompare import runner
    from tscompare.models.registry import build
    from tscompare.tasks import make_synthetic_task
    tasks = [make_synthetic_task(name=f"synthetic/{i}", seed=i, n_series=12, horizon=48) for i in range(4)]
    for m in ["seasonal_naive", "seasonal_avg", "drift"]:
        runner.run(build(m), tasks, out_dir=RES, save_predictions=True, verbose=False, overwrite=True)

## Probe 1 · Error on high-curvature steps vs the rest
Ratio > 1 ⇒ the method is disproportionately worse where the series bends sharply.

In [ ]:
from tscompare.tasks import make_synthetic_task
rows = []
for npz in Path(RES).rglob("*.npz"):
    model, slug = npz.parent.name, npz.stem
    meta = json.loads(npz.with_suffix(".json").read_text())
    # rebuild the identical (seeded) task to get targets + scale; for TIME: re-load via TimeBenchSource with the same caps/seed
    # (the demo regenerates synthetic tasks)
    i = int(slug.split("__")[1]); t = make_synthetic_task(name=f"synthetic/{i}", seed=i, n_series=12, horizon=48) if RES == "/tmp/probe_demo" else None
    if t is None: continue
    rows.append(dict(model=model, task=slug, **error_by_regime(t.targets, np.load(npz)["point"], t.mase_scale)))
probe = pd.DataFrame(rows); probe.groupby("model")[["err_high_curv", "err_low_curv", "ratio"]].mean().round(3)

## Probe 2 · Context-length sensitivity (main table is at 512)
Re-run chosen models with `--context-length 96 336 2048` (models that support it) and compare `mase_rel`. Native TSFMs usually gain from longer context;
patch-LLM variants are bounded by their positional table. Use `--out results_ctx<L>` per setting, then:

In [ ]:
for L in [96, 336, 512, 2048]:
    p = Path(f"results_ctx{L}") if L != 512 else Path("results")
    if p.exists() and any(p.rglob("*.json")):
        print(L, ag.summary_table(ag.load_results(p))["mase_rel"].round(3).to_dict())

## Probe 3 · Patch-size sweep (preview of the micro-patching hypothesis)
`TorchSupervised` exposes patch length/stride (`P`, `S`). Train the same backbone with 16/8 (standard), 8/4, 4/2 (micro-patches) and compare.
Cost note: smaller patches ⇒ more tokens (L=512, P=4,S=2 ⇒ 256 tokens), so run on short-term tasks first. Names must differ so results don't overwrite:

In [ ]:
SWEEP = [(16, 8), (8, 4), (4, 2)]
# from tscompare.models.registry import build; from tscompare import runner
# for P, S in SWEEP:
#     m = build("tan_llm2trsf", P=P, S=S); m.name = f"tan_llm2trsf_P{P}S{S}"; runner.run(m, tasks, out_dir="results_patch", save_predictions=True)
# then aggregate with ag.summary_table(ag.load_results("results_patch")) and run Probe 1 on each variant.
print(SWEEP)